# Phase 2: Paper Parser

This notebook demonstrates the `PaperParser` — loading a scientific publication
from a PMID, PMCID, DOI, or PDF and producing a structured `Paper` object.

**Prerequisites:** Set `ANTHROPIC_API_KEY` in your environment (required for
LLM-assisted parsing of PDFs and URLs; not needed for PMCID/PMID sources
that resolve to structured PMC XML).

In [ ]:
import json, os
from researcher_ai.parsers.paper_parser import PaperParser
from researcher_ai.utils.pdf import (
    extract_figure_ids_from_text,
    split_text_into_sections,
    detect_section_boundaries,
)
from researcher_ai.utils.pubmed import (
    fetch_article_xml, fetch_pmc_fulltext,
    parse_pubmed_xml, parse_jats_xml,
    resolve_doi_to_pmid, resolve_pmid_to_pmcid,
)
from researcher_ai.utils.llm import LLMCache

# Use a local cache to avoid repeated API calls during development
CACHE_DIR = ".llm_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

print('researcher_ai.parsers.paper_parser imported successfully')

## 1. Source type auto-detection

The parser detects source type from the string — no explicit type needed.

In [ ]:
from researcher_ai.models.paper import PaperSource

parser = PaperParser(cache_dir=CACHE_DIR)

examples = [
    "26971820",
    "PMC4878918",
    "10.1038/nmeth.3810",
    "https://doi.org/10.1038/nmeth.3810",
    "https://www.nature.com/articles/nmeth.3810",
    "/path/to/paper.pdf",
]

for src in examples:
    detected = parser._detect_source_type(src)
    print(f"{src[:50]:<52} → {detected.value}")

## 2. Fetch and parse PubMed metadata (PMID)

Uses the NCBI eutils API — no API key required for low-volume access.

In [ ]:
# Fetch raw PubMed XML for the eCLIP paper
PMID = "26971820"  # Van Nostrand et al. 2016 — eCLIP

xml = fetch_article_xml(PMID)
meta = parse_pubmed_xml(xml)

print(f"Title:   {meta['title']}")
print(f"Authors: {meta['authors']}")
print(f"PMID:    {meta['pmid']}")
print(f"PMCID:   {meta.get('pmcid')}")
print(f"DOI:     {meta.get('doi')}")
print(f"Journal: {meta.get('journal')} ({meta.get('year')})")
print(f"\nAbstract (first 200 chars): {meta['abstract'][:200]}...")

## 3. Fetch PMC full-text JATS XML (PMCID)

PMC provides structured JATS XML for open-access articles — rich section
data without needing LLM parsing.

In [ ]:
PMCID = meta.get('pmcid', 'PMC4878918')
print(f"Fetching full text for {PMCID}...")

jats_xml = fetch_pmc_fulltext(PMCID)
jats = parse_jats_xml(jats_xml)

print(f"\nSections found ({len(jats['sections'])}):", [s['title'] for s in jats['sections']])
print(f"Figure captions ({len(jats['figure_captions'])}): {list(jats['figure_captions'].keys())}")
print(f"References ({len(jats['references'])}): first ref = {jats['references'][0] if jats['references'] else 'none'}")
print(f"\nFirst section ({jats['sections'][0]['title']}) text excerpt:")
print(jats['sections'][0]['text'][:400] + '...' if jats['sections'] else 'No sections')

## 4. Utility functions: figure ID extraction and section detection

These regex-based utilities work on any plain text without LLM calls.

In [ ]:
# Extract all figure IDs from a JATS section
full_text = " ".join(s['text'] for s in jats['sections'])
figure_ids = extract_figure_ids_from_text(full_text)
print(f"Figure IDs found in body text: {figure_ids}")

# Section boundaries via regex
sample_text = """
Abstract
eCLIP is a method for identifying RNA-binding protein binding sites.

Introduction
RNA-binding proteins regulate gene expression (Fig. 1).

Results
We identified peaks (Figure 2A). See Supplementary Figure S1 for QC.

Methods and Materials
Cells were grown in DMEM.

References
[1] Ule et al. Science 2003.
"""

bounds = detect_section_boundaries(sample_text)
print(f"\nSection boundaries: {[(t, o) for t, o in bounds]}")

sections = split_text_into_sections(sample_text)
print(f"\nSections split ({len(sections)}):")
for title, text in sections.items():
    print(f"  [{title}] {text[:60].strip()!r}...")

In [ ]:
# Figure ID extraction edge cases
test_cases = [
    "See Figure 1 for an overview.",
    "As shown in Fig. 2A and Fig. 2B.",
    "Figs. 3–5 show the results.",
    "Supplementary Figure S1 shows QC.",
    "See Supplementary Fig. S2A for details.",
    "No figures mentioned here.",
]

for text in test_cases:
    ids = extract_figure_ids_from_text(text)
    print(f"{text!r:<55} → {ids}")

## 5. Full parse via PaperParser

The `PaperParser.parse()` method handles source detection, fetching,
XML parsing, and LLM enrichment in one call.

In [ ]:
# Parse from PMID — will resolve to PMCID and use structured JATS XML.
# LLM is used only for paper_type classification.
# Comment out if ANTHROPIC_API_KEY is not set and use the PMCID path instead.

parser = PaperParser(cache_dir=CACHE_DIR)
paper = parser.parse("26971820")   # eCLIP paper

print(f"Title:       {paper.title}")
print(f"Authors:     {paper.authors}")
print(f"DOI:         {paper.doi}")
print(f"PMID:        {paper.pmid}")
print(f"PMCID:       {paper.pmcid}")
print(f"Paper type:  {paper.paper_type.value}")
print(f"Sections:    {[s.title for s in paper.sections]}")
print(f"Figure IDs:  {paper.figure_ids}")
print(f"References:  {len(paper.references)} total")

In [ ]:
# Inspect the Methods section
methods = paper.methods_section
if methods:
    print(f"Methods section: {methods.title!r}")
    print(f"Figures referenced: {methods.figures_referenced}")
    print(f"\nText excerpt (first 400 chars):")
    print(methods.text[:400])
else:
    print("No methods section found (abstract-only parse).")

In [ ]:
# JSON round-trip: verify the Paper model is fully serializable
json_str = paper.model_dump_json(indent=2)
paper_restored = type(paper).model_validate_json(json_str)

assert paper_restored.title == paper.title
assert paper_restored.doi == paper.doi
assert len(paper_restored.sections) == len(paper.sections)
assert len(paper_restored.figure_ids) == len(paper.figure_ids)

print(f"JSON round-trip: OK ({len(json_str):,} chars)")
print(f"\nFull JSON (first 1000 chars):")
print(json_str[:1000])

## 6. DOI resolution chain

Demonstrates the DOI → PMID → PMCID resolution chain.

In [ ]:
doi = "10.1038/nmeth.3810"

pmid = resolve_doi_to_pmid(doi)
print(f"DOI {doi} → PMID: {pmid}")

if pmid:
    pmcid = resolve_pmid_to_pmcid(pmid)
    print(f"PMID {pmid} → PMCID: {pmcid}")

## 7. LLM cache inspection

In [ ]:
cache = LLMCache(CACHE_DIR)
print(f"Cached LLM responses: {len(cache)}")

# List cache files
from pathlib import Path
cache_files = list(Path(CACHE_DIR).glob('*.json'))
print(f"Cache files: {[f.name for f in cache_files[:5]]}{'...' if len(cache_files)>5 else ''}")

## 8. Parse a second paper

Vanviver et al. 2020 — a multi-omic paper to confirm `paper_type` classification works.

In [ ]:
# ENCODE eCLIP uniform processing pipeline paper — should classify as computational
# PMID: 30423142 (Van Nostrand et al. 2020 — a large-scale atlas, multi-omic)

parser2 = PaperParser(cache_dir=CACHE_DIR)
paper2 = parser2.parse("30423142")

print(f"Title:      {paper2.title[:80]}")
print(f"Paper type: {paper2.paper_type.value}")
print(f"Sections:   {[s.title for s in paper2.sections]}")
print(f"Figures:    {paper2.figure_ids}")
print(f"Refs:       {len(paper2.references)}")

## Summary

Phase 2 delivers:

- `PaperParser.parse(source)` — one call, any source type
- Auto-detection of PMID / PMCID / DOI / PDF / URL
- Structured JATS XML parsing for PMC open-access articles (no LLM needed)
- LLM-assisted section segmentation and metadata extraction for PDFs / URLs
- `extract_figure_ids_from_text()` — regex, handles Fig./Figure/Supplementary
- `split_text_into_sections()` — regex-first section segmentation
- `parse_pubmed_xml()` / `parse_jats_xml()` — structured XML parsers
- `LLMCache` — file-based cache to avoid redundant API calls
- Full JSON round-trip on the resulting `Paper` object

Next: **Phase 3 — Figure Parser** (extract structured `Figure` objects from a parsed `Paper`).